# Namibia Station Audit — GHCND Data Quality Assessment
**Author:** Wilka Igulu  
**Date:** June 2026  
**Repository:** github.com/wilkaigulu/namibia-station-monitor

## Purpose
This notebook audits every Namibian station record available in the
NOAA Global Historical Climatology Network Daily (GHCND) archive.
It computes a Data Quality Score (DQS) for each station and
produces a frozen summary CSV used as the citable dataset for
Igulu (2026).

## Data
Clean daily records are in `audit/data/clean/`. Raw records are
not included in this public repo. Data was retrieved from NOAA
GHCND on 29 June 2026.

In [2]:
# ── Imports and configuration ──
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
CLEAN_DIR = Path("data/clean")
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Station registry — GHCND IDs and known record start years
STATIONS = {
    "windhoek_main" : {"ghcnd": "WA007401540", "name": "Windhoek (GSN)",        "record_start": 1970},
    "windhoek_a"    : {"ghcnd": "WA007400630", "name": "Windhoek (alt A)",      "record_start": 1980},
    "windhoek_b"    : {"ghcnd": "WA007401240", "name": "Windhoek (alt B)",      "record_start": 1980},
    "windhoek_c"    : {"ghcnd": "WA007401850", "name": "Windhoek (alt C)",      "record_start": 1980},
    "gobabis"       : {"ghcnd": "WA007878380", "name": "Gobabis",               "record_start": 1970},
    "keetmanshoop"  : {"ghcnd": "WA004192150", "name": "Keetmanshoop",          "record_start": 1980},
    "mariental"     : {"ghcnd": "WA005688170", "name": "Mariental",             "record_start": 1980},
    "rundu"         : {"ghcnd": "WA012084750", "name": "Rundu",                 "record_start": 1970},
    "ondangwa"      : {"ghcnd": "WAM00068006", "name": "Ondangwa",              "record_start": 1980},
    "walvis_bay"    : {"ghcnd": "WAM00068098", "name": "Walvis Bay",            "record_start": 1980},
}

# Lüderitz — documented absence
LUDERITZ_NOTE = {
    "station_key"   : "luderitz",
    "ghcnd_id"      : "NOT IN GHCND",
    "name"          : "Lüderitz",
    "record_start"  : None,
    "total_records" : 0,
    "record_years"  : 0,
    "completeness"  : 0.0,
    "gap_penalty"   : 1.0,
    "dqs"           : 0.0,
    "usable"        : False,
    "notes"         : "Absent from GHCND entirely. WMO OSCAR lists station as active."
}

REFERENCE_YEARS = 56   # 1970 to 2026
NORMAL_START    = 1981
NORMAL_END      = 2010

print("Configuration loaded.")
print(f"Clean data directory: {CLEAN_DIR.resolve()}")
print(f"Stations to audit: {len(STATIONS)} + Lüderitz (absent)")

Configuration loaded.
Clean data directory: C:\Users\wilka\Documents\namibia-station-monitor\audit\data\clean
Stations to audit: 10 + Lüderitz (absent)


In [3]:
# ── DQS computation ──
def compute_dqs(df, station_key, meta):
    """
    Data Quality Score v1.0
    Three components, each in [0, 1]:
      L — record length relative to 56-year reference horizon
      C — completeness (fraction of non-missing days within record span)
      G — gap penalty (penalises multi-year gaps in the 1981-2010 normal period)
    DQS = 0.4*L + 0.4*C + 0.2*G
    """
    df = df.copy()
    df['DATE'] = pd.to_datetime(df['DATE'])
    df = df.sort_values('DATE')

    # ── L: record length ──
    year_min = df['DATE'].dt.year.min()
    year_max = df['DATE'].dt.year.max()
    record_years = year_max - year_min + 1
    L = min(record_years / REFERENCE_YEARS, 1.0)

    # ── C: completeness ──
    expected_days = (df['DATE'].max() - df['DATE'].min()).days + 1
    actual_days   = len(df)
    C = actual_days / expected_days if expected_days > 0 else 0.0

    # ── G: gap penalty in 1981-2010 normal period ──
    normal = df[
        (df['DATE'].dt.year >= NORMAL_START) &
        (df['DATE'].dt.year <= NORMAL_END)
    ].copy()

    if len(normal) == 0:
        G = 0.0
    else:
        normal['year'] = normal['DATE'].dt.year
        years_present  = normal['year'].nunique()
        years_expected = NORMAL_END - NORMAL_START + 1
        G = years_present / years_expected

    # ── DQS ──
    dqs = round(0.4*L + 0.4*C + 0.2*G, 4)

    return {
        "station_key"  : station_key,
        "ghcnd_id"     : meta["ghcnd"],
        "name"         : meta["name"],
        "record_start" : int(year_min),
        "record_end"   : int(year_max),
        "record_years" : int(record_years),
        "total_records": int(actual_days),
        "completeness" : round(C, 4),
        "normal_coverage": round(G, 4),
        "length_score" : round(L, 4),
        "dqs"          : dqs,
        "usable"       : dqs >= 0.70,
        "notes"        : ""
    }

# ── Run DQS for all stations ──
results = []

for key, meta in STATIONS.items():
    filepath = CLEAN_DIR / f"{key}_daily_clean.csv"
    if not filepath.exists():
        print(f"  ⚠ File not found: {filepath}")
        continue
    df = pd.read_csv(filepath)
    result = compute_dqs(df, key, meta)
    results.append(result)
    print(f"  ✓ {meta['name']:25s}  DQS = {result['dqs']:.4f}  usable = {result['usable']}")

# Add Lüderitz manually
results.append(LUDERITZ_NOTE)
print(f"  ✗ Lüderitz                    DQS = 0.0000  absent from GHCND")
print(f"\n{len(results)} stations processed.")

  ✓ Windhoek (GSN)             DQS = 0.9870  usable = True
  ✓ Windhoek (alt A)           DQS = 0.5334  usable = False
  ✓ Windhoek (alt B)           DQS = 0.5478  usable = False
  ✓ Windhoek (alt C)           DQS = 0.5403  usable = False
  ✓ Gobabis                    DQS = 0.8099  usable = True
  ✓ Keetmanshoop               DQS = 0.5263  usable = False
  ✓ Mariental                  DQS = 0.5484  usable = False
  ✓ Rundu                      DQS = 0.9338  usable = True
  ✓ Ondangwa                   DQS = 0.5849  usable = False
  ✓ Walvis Bay                 DQS = 0.5566  usable = False
  ✗ Lüderitz                    DQS = 0.0000  absent from GHCND

11 stations processed.


In [4]:
# ── Building summary dataframe and printing the finding table ───────────────────
df_results = pd.DataFrame(results)

# Reordering columns for readability
cols = [
    "station_key", "name", "ghcnd_id",
    "record_start", "record_end", "record_years",
    "total_records", "completeness", "normal_coverage",
    "length_score", "dqs", "usable", "notes"
]
df_results = df_results[cols]

# ── Printing the audit table ─────────────────────────────────────────────
print("=" * 70)
print("NAMIBIA GHCND STATION AUDIT — 29 JUNE 2026")
print("=" * 70)
print(f"\n{'Station':<25} {'GHCND ID':<16} {'DQS':>6}  {'Usable':>6}  {'Records':>8}  {'Complete':>9}")
print("-" * 70)

for _, row in df_results.sort_values("dqs", ascending=False).iterrows():
    usable_str   = "YES" if row["usable"] else "NO"
    records_str  = str(int(row["total_records"])) if row["total_records"] > 0 else "—"
    complete_str = f"{row['completeness']*100:.1f}%" if row["completeness"] > 0 else "—"
    print(f"  {row['name']:<23} {row['ghcnd_id']:<16} {row['dqs']:>6.4f}  {usable_str:>6}  {records_str:>8}  {complete_str:>9}")

print("-" * 70)
print(f"\nUsable stations (DQS ≥ 0.70): {df_results['usable'].sum()} of {len(df_results)}")
print(f"Absent from GHCND: 1 (Lüderitz)")
print(f"\nAudit date: 29 June 2026")
print(f"Source: NOAA GHCND (Menne et al., 2012)")

NAMIBIA GHCND STATION AUDIT — 29 JUNE 2026

Station                   GHCND ID            DQS  Usable   Records   Complete
----------------------------------------------------------------------
  Windhoek (GSN)          WA007401540      0.9870     YES     19793      98.5%
  Rundu                   WA012084750      0.9338     YES     17121      85.2%
  Gobabis                 WA007878380      0.8099     YES     12242      60.9%
  Ondangwa                WAM00068006      0.5849      NO      6642      35.0%
  Walvis Bay              WAM00068098      0.5566      NO      7429      58.3%
  Mariental               WA005688170      0.5484      NO      5769      96.7%
  Windhoek (alt B)        WA007401240      0.5478      NO      5539      98.4%
  Windhoek (alt C)        WA007401850      0.5403      NO      5175      98.3%
  Windhoek (alt A)        WA007400630      0.5334      NO      5084      96.6%
  Keetmanshoop            WA004192150      0.5263      NO      5236      93.0%
  Lüderitz      

In [11]:
# ── Exporting frozen audit results ────────────────────────────────────────
output_path = OUTPUT_DIR / "station_audit_results_2026-06.csv"
df_results.to_csv(output_path, index=False)

print(f"Frozen audit results saved to:")
print(f"  {output_path.resolve()}")
print(f"\nRows : {len(df_results)}")
print(f"Cols : {list(df_results.columns)}")
print(f"\nThis file is the citable dataset for Igulu (2026).")
print(f"Do not overwrite, append a new dated file for future audits.")

Frozen audit results saved to:
  C:\Users\wilka\Documents\namibia-station-monitor\audit\data\station_audit_results_2026-06.csv

Rows : 11
Cols : ['station_key', 'name', 'ghcnd_id', 'record_start', 'record_end', 'record_years', 'total_records', 'completeness', 'normal_coverage', 'length_score', 'dqs', 'usable', 'notes']

This file is the citable dataset for Igulu (2026).
Do not overwrite, append a new dated file for future audits.


In [10]:
# ── Key findings memo ──────────────────────────────────────────────────
memo = """
╔══════════════════════════════════════════════════════════════════════╗
║         NAMIBIA STATION AUDIT — KEY FINDINGS                        ║
║         Wilka Igulu · June 2026 · github.com/wilkaigulu             ║
╚══════════════════════════════════════════════════════════════════════╝

AUDIT SCOPE
  Archive  : NOAA Global Historical Climatology Network Daily (GHCND)
  Region   : Namibia
  Date     : 29 June 2026
  Stations : 10 resolvable + 1 confirmed absent (Lüderitz)
  Reference: Menne et al. (2012)

FINDING 1 — Three of eleven stations carry usable long records
  Windhoek GSN  DQS 0.9870  19,793 records  98.5% complete
  Rundu         DQS 0.9338  17,121 records  85.2% complete
  Gobabis       DQS 0.8099  12,242 records  60.9% complete

  The remaining seven stations fall below the DQS 0.70 usability
  threshold. Their records are too short, too incomplete, or too
  discontinuous for extreme-value analysis.

FINDING 2 — Ondangwa: 35% completeness, 1990s missing
  Ondangwa is the gateway station for Namibia's most densely
  populated region. Its GHCND record covers 35.0% of expected
  observation days. The entire 1990s decade is absent from the
  archive.

FINDING 3 — Walvis Bay: 6.6% precipitation coverage since 1990
  The national port and primary industrial corridor has precipitation
  records for 6.6% of days since 1990. Overall record completeness
  is 58.3% but precipitation data specifically post-1990 is
  effectively absent.

FINDING 4 — Lüderitz: absent from GHCND entirely
  Lüderitz is listed as an active station in WMO OSCAR/Surface.
  It does not appear in GHCND. Not sparse. Not gappy. Absent.

  Any hazard estimate a global model produces for the southern
  Namibian coast is derived from stations hundreds of kilometres
  away and from model physics. The deliverable will not say so.

DQS THRESHOLD NOTE
  DQS ≥ 0.70 is the usability threshold for extreme-value analysis
  in the SACRF framework (Igulu, 2025). Stations below this threshold
  receive substantially higher prior weight in Bayesian inference,
  reflecting genuine data scarcity rather than treating sparse records
  as equivalent to long ones.

CITATION
  Igulu, W. (2026). Namibia Station Monitor: GHCND audit and Data
  Quality Scoring for Namibian climate stations (audit-2026-06).
  Zenodo. https://doi.org/10.5281/zenodo.21229782
"""
print(memo)


╔══════════════════════════════════════════════════════════════════════╗
║         NAMIBIA STATION AUDIT — KEY FINDINGS                        ║
║         Wilka Igulu · June 2026 · github.com/wilkaigulu             ║
╚══════════════════════════════════════════════════════════════════════╝

AUDIT SCOPE
  Archive  : NOAA Global Historical Climatology Network Daily (GHCND)
  Region   : Namibia
  Date     : 29 June 2026
  Stations : 10 resolvable + 1 confirmed absent (Lüderitz)
  Reference: Menne et al. (2012)

FINDING 1 — Three of eleven stations carry usable long records
  Windhoek GSN  DQS 0.9870  19,793 records  98.5% complete
  Rundu         DQS 0.9338  17,121 records  85.2% complete
  Gobabis       DQS 0.8099  12,242 records  60.9% complete

  The remaining seven stations fall below the DQS 0.70 usability
  threshold. Their records are too short, too incomplete, or too
  discontinuous for extreme-value analysis.

FINDING 2 — Ondangwa: 35% completeness, 1990s missing
  Ondangwa is th